# Criação da camada Bronze

## Objetivo

Este notebook realiza a ingestão dos arquivos fictícios armazenados na área Landing para a camada Bronze do projeto **Análise de Varejo com Databricks**.

A camada Bronze representa os dados próximos ao formato em que foram recebidos das fontes.

Nesta etapa:

- os dados não são corrigidos;
- registros duplicados são preservados;
- valores nulos são preservados;
- inconsistências textuais são preservadas;
- os campos provenientes dos arquivos são mantidos como texto;
- são adicionadas informações técnicas sobre a origem e o momento da ingestão.

Os dados são armazenados como tabelas Delta no schema `varejo_bronze`.

## Fluxo

Landing (CSV) → Bronze (Delta)

## Processo

Os arquivos CSV armazenados na Landing foram ingeridos e registrados como tabelas Delta no Unity Catalog.

A camada Bronze preservou propositalmente as características dos dados de origem, incluindo problemas de qualidade previamente introduzidos nos dados fictícios.

Também foram adicionados metadados técnicos para possibilitar rastreabilidade das cargas:

- arquivo de origem;
- caminho do arquivo;
- data de modificação do arquivo;
- fonte de dados;
- data e hora da ingestão.

Nenhuma regra de negócio ou tratamento de qualidade foi aplicado nesta camada.

In [0]:
# Bibliotecas utilizadas

from pyspark.sql import functions as F


# ---------------------------------------------------------
# CONFIGURAÇÃO DO AMBIENTE
# ---------------------------------------------------------

catalogo_atual = spark.sql(
    "SELECT current_catalog()"
).first()[0]


schema_landing = "varejo_landing"
schema_bronze = "varejo_bronze"
volume = "dados_brutos"


caminho_volume = (
    f"/Volumes/"
    f"{catalogo_atual}/"
    f"{schema_landing}/"
    f"{volume}"
)


print(f"Catálogo: {catalogo_atual}")
print(f"Schema Bronze: {schema_bronze}")
print(f"Volume de origem: {caminho_volume}")

In [0]:
# ---------------------------------------------------------
# FONTES DE DADOS
# ---------------------------------------------------------

fontes = [
    "clientes",
    "produtos",
    "vendedores",
    "fornecedores",
    "vendas",
    "compras",
    "estoque",
    "contas_receber",
    "contas_pagar",
    "metas_vendas"
]


print(f"Quantidade de fontes: {len(fontes)}")

In [0]:
# ---------------------------------------------------------
# LEITURA DOS ARQUIVOS DA LANDING
# ---------------------------------------------------------

def ler_fonte_landing(nome_fonte):
    """
    Lê uma fonte CSV armazenada na Landing.

    Os campos provenientes da fonte são mantidos
    como STRING para preservar o conteúdo original.

    Também são adicionados metadados técnicos
    de ingestão.
    """

    caminho_fonte = (
        f"{caminho_volume}/{nome_fonte}"
    )


    df = (
        spark.read

        .format("csv")

        .option(
            "header",
            "true"
        )

        .option(
            "inferSchema",
            "false"
        )

        .option(
            "mode",
            "PERMISSIVE"
        )

        .load(
            caminho_fonte
        )
    )


    # Acrescenta informações técnicas da origem antes que o DataFrame perca acesso ao metadado especial do arquivo.

    df = (
        df

        .select(
            "*",

            F.col(
                "_metadata.file_name"
            ).alias(
                "_arquivo_origem"
            ),

            F.col(
                "_metadata.file_path"
            ).alias(
                "_caminho_origem"
            ),

            F.col(
                "_metadata.file_modification_time"
            ).alias(
                "_data_modificacao_arquivo"
            )
        )

        .withColumn(
            "_nome_fonte",
            F.lit(
                nome_fonte
            )
        )

        .withColumn(
            "_data_ingestao",
            F.current_timestamp()
        )
    )


    return df

In [0]:
df_clientes_bronze = ler_fonte_landing(
    "clientes"
)


display(
    df_clientes_bronze.limit(20)
)

In [0]:
df_clientes_bronze.printSchema()

In [0]:
display(
    df_clientes_bronze
    .groupBy(
        "estado"
    )
    .count()
    .orderBy(
        "estado"
    )
)

In [0]:
display(
    df_clientes_bronze
    .select(
        "cidade"
    )
    .distinct()
    .orderBy(
        "cidade"
    )
)

In [0]:
# ---------------------------------------------------------
# GRAVAÇÃO DA CAMADA BRONZE
# ---------------------------------------------------------

def salvar_tabela_bronze(
    dataframe,
    nome_fonte
):
    """
    Grava o DataFrame como uma tabela Delta
    gerenciada pelo Unity Catalog.
    """

    nome_tabela = (
        f"{catalogo_atual}."
        f"{schema_bronze}."
        f"{nome_fonte}"
    )


    (
        dataframe

        .write

        .format(
            "delta"
        )

        .mode(
            "overwrite"
        )

        .option(
            "overwriteSchema",
            "true"
        )

        .option(
            "userMetadata",
            (
                "Carga da fonte "
                f"{nome_fonte} "
                "para a camada Bronze"
            )
        )

        .saveAsTable(
            nome_tabela
        )
    )


    print(
        f"Tabela criada: "
        f"{nome_tabela}"
    )

In [0]:
# ---------------------------------------------------------
# INGESTÃO DE TODAS AS FONTES
# ---------------------------------------------------------

for fonte in fontes:

    print(
        f"Iniciando ingestão: {fonte}"
    )


    df_fonte = ler_fonte_landing(
        fonte
    )


    salvar_tabela_bronze(
        df_fonte,
        fonte
    )


    print(
        f"Ingestão concluída: {fonte}"
    )

    print("-" * 50)

In [0]:
# ---------------------------------------------------------
# DESCRIÇÕES DAS TABELAS
# ---------------------------------------------------------

comentarios_tabelas = {

    "clientes":
        "Dados brutos dos clientes  da Distribuidora Horizonte.",

    "produtos":
        "Dados brutos dos produtos comercializados pela Distribuidora Horizonte.",

    "vendedores":
        "Dados brutos da equipe de vendas.",

    "fornecedores":
        "Dados brutos dos fornecedores.",

    "vendas":
        "Itens de vendas preservados no formato de origem.",

    "compras":
        "Itens de compras realizadas junto aos fornecedores.",

    "estoque":
        "Posições diárias de estoque dos produtos.",

    "contas_receber":
        "Títulos de contas a receber originados das vendas.",

    "contas_pagar":
        "Títulos  de contas a pagar originados das compras.",

    "metas_vendas":
        "Metas comerciais mensais por vendedor."
}

In [0]:
for tabela, comentario in comentarios_tabelas.items():

    spark.sql(
        f"""
        COMMENT ON TABLE
        `{catalogo_atual}`.`{schema_bronze}`.`{tabela}`
        IS '{comentario}'
        """
    )


print(
    "Descrições adicionadas às tabelas."
)

In [0]:
display(
    spark.sql(
        f"""
        SHOW TABLES
        IN `{catalogo_atual}`.`{schema_bronze}`
        """
    )
)

In [0]:
# ---------------------------------------------------------
# VALIDAÇÃO DA CARGA
# ---------------------------------------------------------

resumo_bronze = []


for fonte in fontes:

    nome_tabela = (
        f"{catalogo_atual}."
        f"{schema_bronze}."
        f"{fonte}"
    )


    quantidade = (
        spark.table(
            nome_tabela
        )
        .count()
    )


    resumo_bronze.append(
        (
            fonte,
            quantidade
        )
    )


df_resumo_bronze = (
    spark.createDataFrame(
        resumo_bronze,
        [
            "tabela",
            "quantidade_registros"
        ]
    )
)


display(
    df_resumo_bronze
    .orderBy(
        F.desc(
            "quantidade_registros"
        )
    )
)

In [0]:
display(
    spark.sql(
        f"""
        DESCRIBE HISTORY
        `{catalogo_atual}`.`{schema_bronze}`.`vendas`
        """
    )
)